In [ ]:
#@title **<- Install dependencies and Mount Gdrive** { vertical-output: true, display-mode: "form" }
!pip install libtorrent==2.0.11

!pip3 install --no-cache-dir --force-reinstall lbry-libtorrent

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title **<- Click the play button for start download** { vertical-output: true, display-mode: "form" }
#@markdown <br><center><h1><b>TTGD</b></h1></ceneter>
magnetLinks = "" #@param {type:"string"}
gdriveDownloadPath = "/content/drive/Shareddrives/.trash001" #@param {type:"string"}
compressionMethod = "none" #@param ["none", "TAR", "TAR.XZ"]

#@markdown <br><h1><b>Read Me<b></h1>
#@markdown - To download multiple torrents, paste all the magnet links separated by a space " ". Not recomended unless you want to get band from google
#@markdown - TAR compression method
#@markdown  - Lower compression ratio
#@markdown  - Lower compression time
#@markdown - TAR.XZ compression method
#@markdown  - Higher compression ratio
#@markdown  - Higher compression time<br>

import libtorrent as lt
import shutil, string, time, re,  os

magnetLinks = magnetLinks.split(" ")
gdriveDownloadPath = os.path.join("/content/drive/MyDrive", gdriveDownloadPath)
torrentDownloadPath = "/content/Work_space/DTORRENT/"
torrentCompressPath = "/content/Work_space/CTORRENT/"
chars = re.escape(string.punctuation)

if not os.path.isdir(gdriveDownloadPath):
  os.mkdir(gdriveDownloadPath)

if compressionMethod == "TAR":
  cMethod = "tar"
  cExtention = ".tar"
elif compressionMethod == "TAR.XZ":
  cMethod = "xztar"
  cExtention = ".tar.xz"
elif compressionMethod == "none":
  cMethod = "none"
  cExtention = "none"

for magnetLink in magnetLinks:

  downloadConfig = {
      'save_path': torrentDownloadPath,
      'storage_mode': lt.storage_mode_t(2)}

  SESSION = lt.session()
  SESSION.listen_on(6881, 6891)
  DOWNLOADER = lt.add_magnet_uri(SESSION, magnetLink, downloadConfig)
  SESSION.start_dht()

  print("\n[*]Collecting metadata...")

  while not DOWNLOADER.has_metadata():
    time.sleep(1)

  NAME = str(DOWNLOADER.status().name)

  print("\n[*]Metadata collected...")
  print(f"[*]{NAME} downloading started...\n")

  while not DOWNLOADER.status().is_seeding:
    update = DOWNLOADER.status()

    print(f"[!] {round(update.progress*100, 2):5n}% completed | download_rate: {round(update.download_rate/1000000, 2):5n} MB/s | upload_rate: {round(update.upload_rate/1000000, 2):5n} MB/s | peers_connected: {(update.num_peers):3n} | state: {update.state}")

    time.sleep(5)

  print("\n[*]Download completed...")
  print(f"\n[*]Adding files to a {compressionMethod} file...")

  encNAME = str(re.sub(r'['+chars+']', '',(NAME.encode("ascii", "ignore")).decode())).replace(" ", "_")

  shutil.move(f"{torrentDownloadPath}{NAME}", f"{torrentDownloadPath}{encNAME}")

  if cMethod != "none":
    shutil.make_archive(f"{torrentCompressPath}{encNAME}", cMethod, f"{torrentDownloadPath}")

  if cExtention == "none":
    print("\n[*]Moving downloaded files to Gdrive...")
    shutil.move(f"{torrentDownloadPath}{encNAME}", f"{gdriveDownloadPath}/{encNAME}")
  else:
    print("\n[*]Moving compressed files to Gdrive...")
    shutil.move(f"{torrentCompressPath}{encNAME}{cExtention}", f"{gdriveDownloadPath}/{encNAME}{cExtention}")

  print("\n[*]Cleaning the disk...")

  shutil.rmtree(f"{torrentDownloadPath}{encNAME}", ignore_errors=True)

print("\n[*]Finished.")
print("\n[!]Before you leave please make sure to Disconnect and delete the runtime")

In [ ]:
import libtorrent as lt
import shutil, string, time, re, os

# Escolha entre um magnet link ou um arquivo .torrent
inputType = "torrent"  # "magnet" ou "torrent"

magnetLinks = ""  # Se inputType = "magnet"
torrentFilePath = "/content/a.torrent"  # Se inputType = "torrent"

gdriveDownloadPath = "/content/drive/Shareddrives/.trash001"
compressionMethod = "none"  # Opções: "none", "TAR", "TAR.XZ"

# Criando sessão do libtorrent
SESSION = lt.session()
SESSION.listen_on(6881, 6891)

if inputType == "magnet":
    print("\n[*] Adicionando magnet link...")
    downloadConfig = {'save_path': "./downloads/", 'storage_mode': lt.storage_mode_t(2)}
    DOWNLOADER = lt.add_magnet_uri(SESSION, magnetLinks, downloadConfig)
elif inputType == "torrent":
    print("\n[*] Adicionando arquivo .torrent...")
    info = lt.torrent_info(torrentFilePath)
    downloadConfig = lt.add_torrent_params()
    downloadConfig.ti = info
    downloadConfig.save_path = "./downloads/"
    DOWNLOADER = SESSION.add_torrent(downloadConfig)
else:
    raise ValueError("Opção inválida. Escolha 'magnet' ou 'torrent'.")

print("\n[*] Coletando metadata...")
while not DOWNLOADER.has_metadata():
    time.sleep(1)

NAME = DOWNLOADER.status().name
print(f"\n[*] {NAME} começou o download...\n")

while not DOWNLOADER.status().is_seeding:
    status = DOWNLOADER.status()
    print(f"[!] {round(status.progress * 100, 2)}% concluído | Velocidade: {round(status.download_rate / 1e6, 2)} MB/s | Peers: {status.num_peers} | Estado: {status.state}")
    time.sleep(5)

print("\n[*] Download concluído!")

# Mover arquivos para o Google Drive, se necessário
if os.path.exists(f"./downloads/{NAME}"):
    shutil.move(f"./downloads/{NAME}", f"{gdriveDownloadPath}/{NAME}")

print("\n[*] Arquivos salvos em:", gdriveDownloadPath)
print("\n[*] Finalizado.")


# **Find me on**<br>
- Odysee : [@mrtmash](https://odysee.com/$/invite/@mrtmash:d)
- Github : [@mrtmash](https://github.com/mrtmash)

### <br>**Before you leave please make sure to Disconnect and delete the runtime**